# Workflows de datos con LangGraph

Organizaremos la selección y el análisis de documentos mediante estado, nodos y rutas.
Las decisiones se programan en Python; estos ejemplos no necesitan un modelo ni una API key.

## Contenido
1. Diccionarios y `TypedDict`.
2. Construir, visualizar y ejecutar un grafo.
3. Pydantic: estado y validación de resultados.
4. Rutas condicionales.
5. Paralelismo y reducers.
6. Actualizar y elegir una ruta con `Command`.
7. Dos análisis mediante subgrafos.

Esta sesión contiene ejemplos guiados, sin tareas ni entregables.


## 1. Diccionarios y `TypedDict`

En Neo4j, los nodos representan entidades almacenadas. En LangGraph, los nodos
son funciones y las aristas indican qué paso ejecutar después. El **estado** es
la información compartida entre esos pasos.

Un diccionario puede mezclar campos con diferentes tipos. `dict[str, object]`
permite esa mezcla, pero no describe qué claves esperamos ni el tipo de cada una.


In [ ]:
document: dict[str, object] = {"title": "NumPy arrays", "pages": 20}
print(document["title"], document["pages"])


### Describir los campos

`TypedDict` permite indicar claves concretas y sus tipos. La sintaxis parece una
clase, pero sus valores siguen siendo diccionarios: accedemos con `["pages"]`,
no con `.pages`. Por defecto, las claves declaradas deben estar presentes para
cumplir el contrato de tipos.

[Documentación de Python: TypedDict](https://docs.python.org/3/library/typing.html#typing.TypedDict).


In [ ]:
from typing import TypedDict


class DocumentInfo(TypedDict):
    title: str
    pages: int


document_info: DocumentInfo = {"title": "NumPy arrays", "pages": 20}
print(type(document_info))
print(document_info["pages"] + 1)
assert isinstance(document_info, dict)


### Anotación no significa validación durante la ejecución

`TypedDict` describe las claves y sus tipos para herramientas como mypy,
pero no convierte ni valida los datos durante la ejecución. Más adelante
compararemos este comportamiento con Pydantic.


## 2. Construir, visualizar y ejecutar un grafo

Empezaremos con un solo campo: `topic`. Un nodo normaliza el texto y devuelve
una actualización. Los nodos no necesitan devolver el estado completo ni
modificar el objeto que recibieron. Por defecto, actualizar un campo reemplaza
su valor; los campos que no se mencionan conservan el suyo.

[Estado y actualizaciones](https://docs.langchain.com/oss/python/langgraph/use-graph-api#define-and-update-state).


In [ ]:
class TopicState(TypedDict):
    topic: str


def normalize_typed_topic(state: TopicState) -> TopicState:
    print("Normalizing topic")
    return {"topic": state["topic"].strip().lower()}


input_state: TopicState = {"topic": " Retrieval "}
print(normalize_typed_topic(input_state))
assert input_state["topic"] == " Retrieval "


### Registrar y conectar

`StateGraph` recibe el esquema, `add_node` registra una función y `add_edge`
conecta los pasos. `START` y `END` marcan la entrada y la salida; no son funciones
que tengamos que implementar. `compile()` realiza comprobaciones de estructura
y prepara el grafo ejecutable; no transforma nuestras funciones en código máquina.


In [ ]:
from langgraph.graph import END, START, StateGraph

builder = StateGraph(TopicState)
builder.add_node("normalize", normalize_typed_topic)
builder.add_edge(START, "normalize")
builder.add_edge("normalize", END)
graph = builder.compile()


### Ver el grafo antes de ejecutarlo

`get_graph()` permite consultar su estructura y `draw_mermaid_png()` genera la
imagen. El renderizado predeterminado usa Mermaid.Ink y necesita conexión.
La función siguiente muestra texto Mermaid si no puede obtener la imagen;
esto no impide ejecutar el workflow.

[Visualización oficial](https://docs.langchain.com/oss/python/langgraph/use-graph-api#visualize-your-graph).


In [ ]:
from IPython.display import Image, display
from requests import RequestException


def show_graph(graph):
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except (OSError, ValueError, RuntimeError, RequestException) as error:
        print("Diagram unavailable:", error)
        print(graph.get_graph().draw_mermaid())


show_graph(graph)


### Ejecutar

`invoke()` recibe los valores iniciales y devuelve el resultado final. El dibujo
muestra conexiones posibles; esta llamada ejecuta las funciones conectadas.


In [ ]:
result = graph.invoke({"topic": " Retrieval "})
print(result)
assert result["topic"] == "retrieval"


## 3. Pydantic: estado y validación de resultados

### El mismo grafo con `BaseModel`

LangGraph también admite Pydantic como esquema del estado. Dentro del nodo,
leemos atributos como `state.topic`; seguimos devolviendo un diccionario con
los campos que queremos actualizar. La entrada de `invoke()` puede ser un
diccionario y la salida sigue siendo un diccionario.

[Estado con Pydantic](https://docs.langchain.com/oss/python/langgraph/use-graph-api#use-pydantic-models-for-graph-state).


In [ ]:
from pydantic import BaseModel, ValidationError


class TopicModel(BaseModel):
    topic: str


def normalize_model_topic(state: TopicModel) -> dict[str, str]:
    return {"topic": state.topic.strip().lower()}


model_builder = StateGraph(TopicModel)
model_builder.add_node("normalize", normalize_model_topic)
model_builder.add_edge(START, "normalize")
model_builder.add_edge("normalize", END)
model_graph = model_builder.compile()
model_result = model_graph.invoke({"topic": " Retrieval "})
print(model_result)
assert isinstance(model_result, dict)
assert model_result["topic"] == result["topic"]

try:
    model_graph.invoke({"topic": 123})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["type"])
else:
    raise AssertionError("Invalid input was accepted")


### Diseñar el estado del catálogo

Retomamos los documentos de la sesión anterior. La entrada aporta `topic`; los
campos que se calculan después tienen valores predeterminados. `default_factory=list`
crea una lista independiente para cada instancia.

Conservaremos el tema original y luego seleccionaremos documentos, contaremos
sus páginas y construiremos un mensaje. El estado contiene datos del proceso;
no representa un grafo de conocimiento.


In [ ]:
from pydantic import Field
from sample_data import DOCUMENTS, Document

for document in DOCUMENTS:
    print(document)

class WorkflowState(BaseModel):
    topic: str
    original_topic: str = ""
    documents: list[Document] = Field(default_factory=list)
    count: int = 0
    total_pages: int = 0
    message: str = ""


### Validar una salida dentro de un nodo

Un modelo puede describir el **resultado de una operación**, sin ser el estado
completo del grafo. `AnalysisResult` comprueba éxito, error y medidas no negativas.
`strict=True` evita conversiones implícitas en este resultado.


In [ ]:
from pydantic import ConfigDict

class AnalysisResult(BaseModel):
    model_config = ConfigDict(strict=True)

    success: bool
    error: str = ""
    count: int = Field(ge=0)
    total_pages: int = Field(ge=0)


Usar Pydantic como estado no sustituye esta validación explícita. La guía de
LangGraph señala que la validación del esquema ocurre en la entrada al primer
nodo, no como garantía sobre todas las actualizaciones ni sobre la salida.
Cuando necesitemos verificar un resultado concreto, llamaremos al modelo dentro
del nodo. También comprobamos reglas del dominio, como rechazar un tema en blanco.


## 4. Rutas condicionales

Normalizar → seleccionar → resumir o informar que no hubo resultados.
El nodo de selección actualiza `documents` y conserva el resto del estado.

`StateUpdate` es un nombre para `dict[str, object]`: una actualización puede
contener campos con valores de distintos tipos. Así distinguimos el estado
completo de los campos que devuelve cada nodo.



In [ ]:
StateUpdate = dict[str, object]

def normalize_topic(state: WorkflowState) -> StateUpdate:
    topic = state.topic.strip().lower()
    if not topic:
        raise ValueError("topic must not be blank")
    return {"original_topic": state.topic, "topic": topic}

def select_documents(state: WorkflowState) -> StateUpdate:
    selected = []
    for document in DOCUMENTS:
        if document["topic"] == state.topic:
            selected.append(document)
    return {"documents": selected}


### Decidir y producir la salida

La función de ruta devuelve una etiqueta: `found` o `empty`. No actualiza el
estado. El nodo de resumen valida su resultado con `AnalysisResult` antes de
publicar las medidas. Una consulta válida sin coincidencias produce ceros y
un mensaje; no es lo mismo que una entrada inválida.


In [ ]:
from typing import Literal

def route_documents(state: WorkflowState) -> Literal["found", "empty"]:
    if state.documents:
        return "found"
    return "empty"

def summarize_documents(state: WorkflowState) -> StateUpdate:
    raw_result = {
        "success": True,
        "count": len(state.documents),
        "total_pages": sum(document["pages"] for document in state.documents),
    }
    result = AnalysisResult.model_validate(raw_result)
    return {
        "count": result.count,
        "total_pages": result.total_pages,
        "message": f"Found {result.count} documents with {result.total_pages} pages.",
    }

def report_empty(state: WorkflowState) -> StateUpdate:
    return {
        "count": 0,
        "total_pages": 0,
        "message": f"No documents found for topic: {state.topic}",
    }


### Construir las dos rutas

`add_conditional_edges` relaciona la etiqueta con el nodo correspondiente.
Separamos la construcción en una función para reconstruir el grafo cuando
cambiemos funciones o conexiones.


In [ ]:
def build_workflow():
    builder = StateGraph(WorkflowState)
    builder.add_node("normalize", normalize_topic)
    builder.add_node("select", select_documents)
    builder.add_node("summarize", summarize_documents)
    builder.add_node("empty", report_empty)
    builder.add_edge(START, "normalize")
    builder.add_edge("normalize", "select")
    builder.add_conditional_edges(
        "select", route_documents,
        {"found": "summarize", "empty": "empty"},
    )
    builder.add_edge("summarize", END)
    builder.add_edge("empty", END)
    return builder.compile()


In [ ]:
workflow = build_workflow()
show_graph(workflow)
result = workflow.invoke({"topic": " Retrieval "})
empty_result = workflow.invoke({"topic": "unknown"})
print(result["message"])
print(empty_result["message"])
assert result["count"] == 2
assert result["total_pages"] == 28
assert empty_result["count"] == 0

try:
    workflow.invoke({"topic": "   "})
except ValueError as error:
    print("Rejected:", error)
else:
    raise AssertionError("Blank topic was accepted")


## 5. Paralelismo y reducers

### Dos operaciones independientes

Después de seleccionar documentos, una rama cuenta documentos y otra suma
páginas. Cada rama escribe un campo distinto. Un nodo final espera ambas y
construye el mensaje: apertura de ramas (*fan-out*) y reunión (*fan-in*).

Simulamos dos esperas de dos segundos para ver que las tareas se solapan.
Los cálculos pequeños del catálogo no necesitan esas esperas ni obtienen por
sí mismos una ventaja de velocidad; el patrón es útil para operaciones independientes.

[Paralelismo oficial](https://docs.langchain.com/oss/python/langgraph/use-graph-api#run-graph-nodes-in-parallel).


In [ ]:
import asyncio

async def count_documents(state: WorkflowState) -> StateUpdate:
    print("Count started")
    await asyncio.sleep(2)
    print("Count finished")
    return {"count": len(state.documents)}

async def count_pages(state: WorkflowState) -> StateUpdate:
    print("Pages started")
    await asyncio.sleep(2)
    print("Pages finished")
    return {"total_pages": sum(document["pages"] for document in state.documents)}

def format_summary(state: WorkflowState) -> StateUpdate:
    result = AnalysisResult(
        success=True, count=state.count, total_pages=state.total_pages
    )
    return {"message": f"Found {result.count} documents with {result.total_pages} pages."}


Las dos conexiones desde `select` activan ambas ramas. La conexión
`add_edge(["count", "pages"], "format")` exige que ambas terminen antes de
construir el mensaje. Como los nodos son `async`, ejecutamos con `await ...ainvoke()`.


In [ ]:
def build_parallel_workflow():
    builder = StateGraph(WorkflowState)
    builder.add_node("normalize", normalize_topic)
    builder.add_node("select", select_documents)
    builder.add_node("count", count_documents)
    builder.add_node("pages", count_pages)
    builder.add_node("format", format_summary)
    builder.add_edge(START, "normalize")
    builder.add_edge("normalize", "select")
    builder.add_edge("select", "count")
    builder.add_edge("select", "pages")
    builder.add_edge(["count", "pages"], "format")
    builder.add_edge("format", END)
    return builder.compile()


In [ ]:
from time import perf_counter

parallel_workflow = build_parallel_workflow()
show_graph(parallel_workflow)
started = perf_counter()
parallel_result = await parallel_workflow.ainvoke({"topic": "retrieval"})
print(parallel_result["message"])
print("Elapsed seconds:", round(perf_counter() - started, 2))
assert parallel_result["count"] == 2
assert parallel_result["total_pages"] == 28


### Cuando dos ramas escriben el mismo campo

Sin una regla de combinación, dos escrituras al mismo campo en el mismo paso
pueden producir `InvalidUpdateError`. Un **reducer** indica cómo combinar el valor
anterior con la actualización. Para listas, `operator.add` concatena; no suma
sus elementos numéricamente.

`Annotated[list[str], add]` conserva el tipo y asocia la función de combinación
que LangGraph utilizará. `Field(default_factory=list)` aporta la lista inicial.

[Reducers](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers).


In [ ]:
from operator import add
from typing import Annotated

print(add(["Count finished"], ["Pages finished"]))
class EventState(WorkflowState):
    events: Annotated[list[str], add] = Field(default_factory=list)

def record_count(state: EventState) -> StateUpdate:
    return {"events": [f"Count: {len(state.documents)}"]}

def record_pages(state: EventState) -> StateUpdate:
    total = sum(document["pages"] for document in state.documents)
    return {"events": [f"Pages: {total}"]}

def build_event_workflow():
    builder = StateGraph(EventState)
    builder.add_node("count", record_count)
    builder.add_node("pages", record_pages)
    builder.add_edge(START, "count")
    builder.add_edge(START, "pages")
    builder.add_edge("count", END)
    builder.add_edge("pages", END)
    return builder.compile()


In [ ]:
event_workflow = build_event_workflow()
event_result = event_workflow.invoke({"topic": "retrieval", "documents": result["documents"]})
print(event_result["events"])
assert set(event_result["events"]) == {"Count: 2", "Pages: 28"}


## 6. Actualizar y elegir una ruta con `Command`

Antes separábamos selección y decisión de ruta. `Command` permite que un nodo
actualice `documents` y elija `summarize` o `empty` en su retorno.
`update` contiene los cambios y `goto` el destino. `Literal` declara destinos
posibles para el tipado y el dibujo.

Esta es una variante del mismo análisis. No añadimos conexiones fijas ni una
ruta condicional desde `select`: esas conexiones también se ejecutarían y no
quedan canceladas por `goto`.

[Command](https://docs.langchain.com/oss/python/langgraph/graph-api#command).


In [ ]:
from langgraph.types import Command

def select_and_route(
    state: WorkflowState,
) -> Command[Literal["summarize", "empty"]]:
    selected = []
    for document in DOCUMENTS:
        if document["topic"] == state.topic:
            selected.append(document)
    destination = "summarize" if selected else "empty"
    return Command(update={"documents": selected}, goto=destination)

def build_command_workflow():
    builder = StateGraph(WorkflowState)
    builder.add_node("normalize", normalize_topic)
    builder.add_node("select", select_and_route)
    builder.add_node("summarize", summarize_documents)
    builder.add_node("empty", report_empty)
    builder.add_edge(START, "normalize")
    builder.add_edge("normalize", "select")
    builder.add_edge("summarize", END)
    builder.add_edge("empty", END)
    return builder.compile()


In [ ]:
command_workflow = build_command_workflow()
show_graph(command_workflow)
for topic in ("retrieval", "unknown"):
    command_result = command_workflow.invoke({"topic": topic})
    print(command_result["message"])
    assert command_result["message"] == workflow.invoke({"topic": topic})["message"]


## 7. Dos análisis mediante subgrafos

Un **subgrafo** reúne varios pasos en un grafo compilado que otro usa como nodo.
Analizaremos todo el catálogo mediante dos ramas: una suma páginas y redacta un
resumen; otra cuenta documentos por tema y redacta otro resumen.

Ambas leen `documents`. Cada una conserva su cálculo intermedio en su propio
estado y devuelve un campo diferente al padre: `pages_report` o `topics_report`.

[Subgrafos y campos compartidos](https://docs.langchain.com/oss/python/langgraph/use-subgraphs#add-a-subgraph-as-a-node).


In [ ]:
class CatalogState(BaseModel):
    documents: list[Document] = Field(default_factory=list)
    pages_report: str = ""
    topics_report: str = ""


class PagesState(CatalogState):
    total_pages: int = 0


class TopicsState(CatalogState):
    topic_counts: dict[str, int] = Field(default_factory=dict)


class PagesOutput(BaseModel):
    pages_report: str = ""


class TopicsOutput(BaseModel):
    topics_report: str = ""


### Los pasos internos

Cada análisis separa el cálculo de la redacción del resultado. `counts.get(topic, 0)`
lee el conteo actual o empieza en cero si el tema todavía no aparece.


In [ ]:
def calculate_pages(state: PagesState) -> StateUpdate:
    total = sum(document["pages"] for document in state.documents)
    return {"total_pages": total}


def write_pages_report(state: PagesState) -> StateUpdate:
    return {"pages_report": f"{len(state.documents)} documents, {state.total_pages} pages"}


def count_topics(state: TopicsState) -> StateUpdate:
    counts: dict[str, int] = {}
    for document in state.documents:
        topic = document["topic"]
        counts[topic] = counts.get(topic, 0) + 1
    return {"topic_counts": counts}


def write_topics_report(state: TopicsState) -> StateUpdate:
    lines = []
    for topic, count in sorted(state.topic_counts.items()):
        lines.append(f"{topic}: {count}")
    return {"topics_report": ", ".join(lines)}


### Compilar los hijos y conectarlos al padre

`output_schema` limita los campos que devuelve cada subgrafo. Así ambos pueden
leer `documents` sin devolverlo ni intentar escribirlo simultáneamente en el padre.
Las dos conexiones desde `prepare_catalog` activan ambos análisis.

[Esquemas de entrada y salida](https://docs.langchain.com/oss/python/langgraph/graph-api#multiple-schemas).


In [ ]:
def build_pages_subgraph():
    builder = StateGraph(PagesState, output_schema=PagesOutput)
    builder.add_node("calculate_pages", calculate_pages)
    builder.add_node("write_report", write_pages_report)
    builder.add_edge(START, "calculate_pages")
    builder.add_edge("calculate_pages", "write_report")
    builder.add_edge("write_report", END)
    return builder.compile()


def build_topics_subgraph():
    builder = StateGraph(TopicsState, output_schema=TopicsOutput)
    builder.add_node("count_topics", count_topics)
    builder.add_node("write_report", write_topics_report)
    builder.add_edge(START, "count_topics")
    builder.add_edge("count_topics", "write_report")
    builder.add_edge("write_report", END)
    return builder.compile()


def prepare_catalog(state: CatalogState) -> StateUpdate:
    return {"documents": state.documents}


def build_modular_workflow():
    builder = StateGraph(CatalogState)
    builder.add_node("prepare_catalog", prepare_catalog)
    builder.add_node("pages_analysis", build_pages_subgraph())
    builder.add_node("topics_analysis", build_topics_subgraph())
    builder.add_edge(START, "prepare_catalog")
    builder.add_edge("prepare_catalog", "pages_analysis")
    builder.add_edge("prepare_catalog", "topics_analysis")
    builder.add_edge("pages_analysis", END)
    builder.add_edge("topics_analysis", END)
    return builder.compile()


### Ver también los pasos internos

`get_graph(xray=1)` expande un nivel de subgrafos: veremos los dos pasos dentro
de cada rama, además de las conexiones del padre. La vista sin `xray` muestra
cada subgrafo como un único nodo.


In [ ]:
modular_workflow = build_modular_workflow()
try:
    display(Image(modular_workflow.get_graph(xray=1).draw_mermaid_png()))
except (OSError, ValueError, RuntimeError, RequestException) as error:
    print("Diagram unavailable:", error)
    print(modular_workflow.get_graph(xray=1).draw_mermaid())

catalog_result = modular_workflow.invoke({"documents": DOCUMENTS})
print(catalog_result["pages_report"])
print(catalog_result["topics_report"])
assert catalog_result["pages_report"] == "6 documents, 108 pages"
assert catalog_result["topics_report"] == "data: 2, integration: 1, python: 1, retrieval: 2"


## Referencias

- [LangGraph: overview](https://docs.langchain.com/oss/python/langgraph/overview).
- [Graph API: estado, nodos, rutas, reducers y Command](https://docs.langchain.com/oss/python/langgraph/graph-api).
- [Estado con Pydantic](https://docs.langchain.com/oss/python/langgraph/use-graph-api#use-pydantic-models-for-graph-state).
- [Visualización](https://docs.langchain.com/oss/python/langgraph/use-graph-api#visualize-your-graph).
- [Paralelismo](https://docs.langchain.com/oss/python/langgraph/use-graph-api#run-graph-nodes-in-parallel).
- [Subgraphs](https://docs.langchain.com/oss/python/langgraph/use-subgraphs).
- [Python: TypedDict](https://docs.python.org/3/library/typing.html#typing.TypedDict).
- [Pydantic: modelos](https://docs.pydantic.dev/latest/concepts/models/).
